In [ ]:
from pathlib import Path
import polars as pl
from dtale import show
import dtale.global_state as global_state

global_state.set_app_settings(dict(max_column_width=300))

data_dir = Path().absolute() / ".." / "data"
df = pl.read_parquet(data_dir / "interests/cm0i27jdj0000aqpa73ghpcxf.snappy")
      # .drop("raw_interests", "raw_results")
      # .explode("interests", "interests_quirkiness")
      # .filter(pl.col("interests_quirkiness")))

d = show(df.to_pandas())
d.open_browser()

In [ ]:
import umap
import plotly.express as px

In [ ]:
df = df.filter(pl.col("fine_centroid").is_not_null()).group_by("fine_dissimilarity_rank").agg(pl.col("fine_centroid").first())

# Reduce dimensionality of fine_centroid embeddings
reducer = umap.UMAP(n_components=2, random_state=42)
embeddings_2d = reducer.fit_transform(df['fine_centroid'].to_list())

# Create a new dataframe with reduced embeddings and labels
plot_df = pl.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'rank': df['fine_dissimilarity_rank']
})

# Create an interactive scatter plot
fig = px.scatter(
    plot_df.to_pandas(),
    x='x',
    y='y',
    color='rank',
    hover_data=['rank'],
    title='Fine Centroid Embeddings Visualization'
)

# Show the plot
fig.show()